# CLV Thesis — Final Kaggle Runner

Runs the **final Valendin CDNOW replication evidence set** across 3 seeds (42, 7, 2024), then builds interpretation tables and diagnostics. Extension sweeps are present later in the notebook but skipped by default.

**CDNOW Valendin replication uses the full 23,570-customer master file via `lstm_base_cdnow_replication`, not the older `*_cdnow_full` configs.** Those older configs are not part of the headline replication path.

### How to run
1. Settings → Accelerator → **GPU T4 x1** (or x2); Internet → **On**.
2. Attach the four Kaggle datasets (`cdnow-dataset` incl. `CDNOW_master.txt`, `uci-retail`, `tafeng-dataset`, `dunnhumby`).
3. **Run All.** Cells are resumable (`--skip_existing`); re-running continues where it stopped.

### Benchmarks
Probabilistic benchmarks (Pareto/NBD, BG/NBD+GG, Pareto/GGG — the latter needs R) are generated **locally** and committed to the repo under `results/tables/`; the comparison cell copies them into the working results. They are **not** retrained here.


## 1 · Setup — install, clone repo, fetch data, GPU check

In [ ]:
import subprocess, sys, os, shutil
from pathlib import Path

# ── 1. Install missing packages (skipped if already present) ─────────────────
# We do NOT touch numpy here. Kaggle's base image ships NumPy 2.x and ~15
# preinstalled packages (shap, jax, cupy, opencv, pytensor, ...) require >=2.0.
# The codebase was audited 2026-05-13 and is NumPy 2.x compatible.
def _need_install(pkg_name):
    try:
        __import__(pkg_name)
        return False
    except ImportError:
        return True

to_install = []
if _need_install("lifetimes"):  to_install.append("lifetimes>=0.11.3")
if _need_install("openpyxl"):   to_install.append("openpyxl>=3.1.0")

if to_install:
    print(f"Installing: {to_install}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + to_install, check=True)
    print("Done.\n")
else:
    print("All packages already installed.\n")

# ── 2. Set Kaggle environment flag ────────────────────────────────────────────
os.environ["KAGGLE_ENV"] = "1"
print("KAGGLE_ENV=1 set.\n")

# ── 3. Clone (or refresh) the repo from GitHub ────────────────────────────────
# Internet must be ON: Settings → Internet → On. Pin a specific commit by
# setting THESIS_REF before running this cell, e.g. os.environ["THESIS_REF"]="<sha>".
REPO_URL  = "https://github.com/OttoPrins/thesis-code-final.git"
REPO_PATH = Path("/kaggle/working/thesis-code")
REPO_REF  = os.environ.get("THESIS_REF", "main")

if REPO_PATH.exists():
    subprocess.run(["git", "-C", str(REPO_PATH), "fetch", "--all", "--tags", "--quiet"], check=True)
    subprocess.run(["git", "-C", str(REPO_PATH), "checkout", REPO_REF, "--quiet"], check=True)
    subprocess.run(["git", "-C", str(REPO_PATH), "reset", "--hard", f"origin/{REPO_REF}", "--quiet"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_PATH)], check=True)

os.chdir(REPO_PATH)
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))
sha = subprocess.check_output(
    ["git", "-C", str(REPO_PATH), "rev-parse", "--short", "HEAD"]
).decode().strip()
print(f"Repo : {REPO_PATH}  @ {sha}  (ref={REPO_REF})")


# ── 3.5. Ensure raw datasets are accessible ────────────────────────────────────
# CDNOW_sample.txt comes from git; CDNOW_master.txt is optional and fetched from Kaggle when available.
# UCI has a UCI ML Repository fallback URL if the Kaggle dataset upload was skipped.
# TaFeng and Dunnhumby come from Kaggle (pre-mounted symlink or kaggle CLI download).
DATA_ROOT = Path("/kaggle/working/input")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# ── CDNOW: sample from git, optional master from Kaggle ──────────────────
cdnow_dst    = DATA_ROOT / "cdnow-dataset"
cdnow_sample = cdnow_dst / "CDNOW_sample.txt"
cdnow_master = cdnow_dst / "CDNOW_master.txt"
cdnow_mount  = Path("/kaggle/input/cdnow-dataset")
cdnow_dst.mkdir(exist_ok=True)

# CDNOW_sample.txt (2,357-customer 10% sample) is tracked in git — copy from the
# repo. Fall back to the Kaggle dataset mount if the repo copy is missing.
if not cdnow_sample.exists():
    repo_sample = REPO_PATH / "data" / "raw" / "CDNOW_sample.txt"
    if repo_sample.exists():
        shutil.copy(repo_sample, cdnow_sample)
        print("  cdnow-dataset: CDNOW_sample.txt copied from git repo.")
    elif (cdnow_mount / "CDNOW_sample.txt").exists():
        shutil.copy(cdnow_mount / "CDNOW_sample.txt", cdnow_sample)
        print("  cdnow-dataset: CDNOW_sample.txt copied from /kaggle/input/.")
    else:
        print("  cdnow-dataset: WARN — CDNOW_sample.txt not found in repo or /kaggle/input/.")
else:
    print("  cdnow-dataset: CDNOW_sample.txt already present.")

# CDNOW_master.txt (23,570-customer cohort) is gitignored — comes from the Kaggle
# dataset only. Needed for the Valendin et al. (2022) 39x39 replication protocol.
if not cdnow_master.exists():
    if (cdnow_mount / "CDNOW_master.txt").exists():
        shutil.copy(cdnow_mount / "CDNOW_master.txt", cdnow_master)
        print("  cdnow-dataset: CDNOW_master.txt copied from /kaggle/input/.")
    else:
        print("  cdnow-dataset: CDNOW_master.txt not in /kaggle/input/ — trying kaggle download ...")
        try:
            subprocess.run(
                ["kaggle", "datasets", "download", "-d", "ottoprins/cdnow-dataset",
                 "-p", str(cdnow_dst), "--unzip", "--force"],
                check=True, capture_output=True, timeout=180,
            )
        except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as e:
            print(f"  cdnow-dataset: kaggle download fallback failed ({type(e).__name__}).")
        if cdnow_master.exists():
            print("  cdnow-dataset: CDNOW_master.txt downloaded from ottoprins/cdnow-dataset.")
        else:
            print("  cdnow-dataset: WARN — CDNOW_master.txt still missing. "
                  "Cell 2.5 will skip; normal thesis sweeps use CDNOW_sample.txt.")
else:
    print("  cdnow-dataset: CDNOW_master.txt already present.")

# ── UCI: Kaggle download with UCI ML Repo fallback ────────────────────────────
uci_dst = DATA_ROOT / "uci-retail"
if uci_dst.exists():
    print("  uci-retail: already present.")
elif Path("/kaggle/input/uci-retail").exists():
    uci_dst.symlink_to(Path("/kaggle/input/uci-retail"))
    print("  uci-retail: symlinked from /kaggle/input/.")
else:
    print("  uci-retail: not mounted — trying kaggle download ...")
    try:
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", "ottoprins/uci-retail",
             "-p", str(uci_dst), "--unzip"],
            check=True, capture_output=True, timeout=300,
        )
        print("  uci-retail: kaggle download complete.")
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as e:
        print(f"  uci-retail: kaggle download failed ({type(e).__name__}) — fetching from UCI ML Repo ...")
        uci_dst.mkdir(exist_ok=True)
        subprocess.run(["wget", "-q", "-O", "/tmp/uci.zip",
            "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"],
            check=True)
        subprocess.run(["unzip", "-q", "-o", "/tmp/uci.zip", "-d", str(uci_dst)],
            check=True)
        # UCI zip may extract with a slightly different name — normalise it
        for f in sorted(uci_dst.iterdir()):
            if f.suffix in (".xlsx", ".csv") and "retail" in f.name.lower():
                target_name = uci_dst / "online_retail_II.xlsx"
                if not target_name.exists():
                    f.rename(target_name)
                break
        print("  uci-retail: UCI fallback download complete.")

# ── TaFeng and Dunnhumby: standard Kaggle download / symlink ─────────────────
for slug, api_ref in [("tafeng-dataset", "ottoprins/tafeng-dataset"),
                       ("dunnhumby",      "ottoprins/dunnhumby")]:
    target  = DATA_ROOT / slug
    mounted = Path(f"/kaggle/input/{slug}")
    if target.exists():
        print(f"  {slug}: already present.")
    elif mounted.exists():
        target.symlink_to(mounted)
        print(f"  {slug}: symlinked from /kaggle/input/.")
    else:
        print(f"  {slug}: not mounted — downloading ...")
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", api_ref,
             "-p", str(target), "--unzip"],
            check=True,
        )
        print(f"  {slug}: download complete.")

# ── Confirm what's on disk ────────────────────────────────────────────────────
print("\n── Data directory contents ──")
for slug in ["cdnow-dataset", "uci-retail", "tafeng-dataset", "dunnhumby"]:
    d = DATA_ROOT / slug
    if d.exists():
        files = sorted(f.name for f in d.iterdir() if not f.name.startswith("."))[:6]
        print(f"  {slug}: {files}")
    else:
        print(f"  {slug}: [MISSING]")

os.environ["KAGGLE_DATA_ROOT"] = str(DATA_ROOT)
print(f"\nKAGGLE_DATA_ROOT={DATA_ROOT}  — all datasets ready.\n")

# ── 4. Verify GPU (fail fast if incompatible) ─────────────────────────────────
import torch
print(f"\ntorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    cap  = torch.cuda.get_device_capability(0)
    print(f"GPU  : {name}  (sm_{cap[0]}{cap[1]})")
    print(f"VRAM : {vram:.1f} GB")
    if cap < (7, 0):
        raise RuntimeError(
            f"\n  {name} has CUDA capability sm_{cap[0]}{cap[1]} "
            f"(PyTorch {torch.__version__} requires sm_70+).\n"
            "  Fix: Settings -> Accelerator -> GPU T4 x1 or T4 x2\n"
            "       then restart the kernel and re-run Cell 1."
        )
    print(f"GPU capability OK (sm_{cap[0]}{cap[1]} >= 7.0).")
else:
    raise RuntimeError(
        "No GPU found. Enable one: Settings -> Accelerator -> GPU T4 x1 (or T4 x2)."
    )

# ── 5. Output directory ───────────────────────────────────────────────────────
Path("/kaggle/working/results").mkdir(parents=True, exist_ok=True)
print("\n/kaggle/working/results/ — ready.")


## 2 · Validate data mounts & pipeline

In [ ]:
import os
from pathlib import Path

# ── Check dataset mounts ──────────────────────────────────────────────────────
# After Cell 1, KAGGLE_DATA_ROOT=/kaggle/working/input — check there.
# On a fresh interactive session without Cell 1 run, falls back to /kaggle/input
# (where Dunnhumby is still pre-mounted from the UI attachment).
data_root = os.environ.get("KAGGLE_DATA_ROOT", "/kaggle/input")

REQUIRED_FILES = {
    "cdnow-dataset" : ["CDNOW_sample.txt", "CDNOW_master.txt"],
    "uci-retail"    : ["online_retail_II.xlsx"],
    "tafeng-dataset": ["ta_feng_all_months_merged.csv"],
    "dunnhumby"     : ["transaction_data.csv", "hh_demographic.csv"],
}
OPTIONAL_FILES = {}  # CDNOW_master.txt is now required (full-cohort protocol)

all_ok = True
optional_ok = True
for slug, expected in REQUIRED_FILES.items():
    p = Path(data_root) / slug
    label = f"{data_root}/{slug}/"
    if not p.exists():
        print(f"✗  {label}  — NOT FOUND  (run Cell 1 first)")
        all_ok = False
        continue
    actual = sorted(f.name for f in p.iterdir())
    missing = [f for f in expected if f not in actual]
    if missing:
        print(f"⚠  {label}  — found but missing: {missing}")
        print(f"   Files present: {actual[:10]}")
        all_ok = False
    else:
        print(f"✓  {label}  — required files present; {len(actual)} file(s): {actual[:5]}")

for slug, expected in OPTIONAL_FILES.items():
    p = Path(data_root) / slug
    label = f"{data_root}/{slug}/"
    if not p.exists():
        optional_ok = False
        print(f"⚠  {label}  — optional files unavailable: {expected}")
        continue
    actual = sorted(f.name for f in p.iterdir())
    missing = [f for f in expected if f not in actual]
    if missing:
        optional_ok = False
        print(f"⚠  {label}  — optional files missing: {missing}")
        print(f"   Cell 2.5 will skip; normal thesis sweeps continue with CDNOW_sample.txt.")
    else:
        print(f"✓  {label}  — optional Valendin master file present.")

print()
if not all_ok:
    raise SystemExit(
        "Missing required Kaggle input files. Run Cell 1 again and make sure the "
        "required Kaggle datasets are mounted or downloadable."
    )
else:
    print("Required datasets ready.")
    if not optional_ok:
        print("Optional Valendin master replication is unavailable; Cell 2.5 will skip.")

# ── Quick pipeline validation (CDNOW only — takes ~10 s) ──────────────────────
# This runs the data pipeline end-to-end and checks tensor shapes.
# It does NOT train a model.
print("\nRunning pipeline validation for CDNOW...")
!python validate_pipelines.py --dataset cdnow

# Uncomment to validate other datasets:
# !python validate_pipelines.py --dataset uci
# !python validate_pipelines.py --dataset tafeng
# !python validate_pipelines.py --dataset dunnhumby


## 3 · Valendin et al. (2022) replication — CDNOW master 39×39

Runs `lstm_base_cdnow_replication` on 3 seeds (42, 7, 2024): strict reference-demo track — 39-week calibration + holdout, full 23,570-customer master, observed softmax classes, `batch_size=32`, no repair/calibration tricks, temperature=1.0.

Reference: Valendin et al. (2022) Table 4 CDNOW row: RMSE = 1.86, bias = −0.7%, MAPE = 13.8%.

**Requires `CDNOW_master.txt` in the `cdnow-dataset` Kaggle dataset.**

In [ ]:
# Stage 0 — Base LSTM CDNOW replication (3 seeds × lstm_base_cdnow_final)
#
# Uses the validated Valendin-winning config from experiments/configs_final/.
# Paper target (Table 4 CDNOW): RMSE=1.86, bias=-0.7%, MAPE=13.8%
import subprocess, sys, os, json, glob
import numpy as np
import pandas as pd
from pathlib import Path

FINAL_CONFIG_DIR = "experiments/configs_final"
SEEDS = [42, 7, 2024]

data_root = Path(os.environ.get("KAGGLE_DATA_ROOT", "/kaggle/input"))
master = data_root / "cdnow-dataset" / "CDNOW_master.txt"
if not master.exists():
    print(f"SKIP: CDNOW_master.txt not found at {master}.")
    print("Ensure ottoprins/cdnow-dataset contains the master file, then re-run.")
else:
    sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip()
    print(f"Replication git SHA: {sha}")

    subprocess.run(
        [sys.executable, "run_seeds.py",
         "--config_dir", FINAL_CONFIG_DIR,
         "--configs", "lstm_base_cdnow_final",
         "--seeds", "42", "7", "2024",
         "--modes", "sample",
         "--skip_existing",
         "--heartbeat_interval", "60"],
        check=False,
    )

    # Print per-seed results
    print("\n" + "="*60)
    print("REPLICATION RESULTS vs Valendin et al. (2022)")
    print("="*60)
    print(f"{'seed':>5} | {'RMSE':>6} | {'bias%':>7} | {'MAPE':>7}")
    print("-"*35)
    seed_results = []
    for seed in SEEDS:
        hits = glob.glob(
            f"/kaggle/working/results/tables/lstm_base_cdnow_final_seed{seed}_sample_metrics.json"
        )
        if hits:
            m = json.load(open(hits[0]))
            r = m.get("freq_rmse", float("nan"))
            b = m.get("bias_pct", float("nan"))
            mp = m.get("freq_valendin_mape", float("nan"))
            seed_results.append((r, b, mp))
            print(f"{seed:>5} | {r:>6.3f} | {b:>+7.1f} | {mp:>7.1f}")
    if seed_results:
        mean = np.array(seed_results).mean(0)
        print(f"{'mean':>5} | {mean[0]:>6.3f} | {mean[1]:>+7.1f} | {mean[2]:>7.1f}")
    print("-"*35)
    print(f"{'paper':>5} |  1.860 |    -0.7 |    13.8")

    # Build ensemble + diagnostics
    print("\nBuilding seed ensembles and interpretation table...")
    subprocess.run(
        [sys.executable, "-m", "src.evaluation.replication_report",
         "--results-dir", "/kaggle/working/results",
         "--build-array-ensembles"],
        check=False,
    )
    report_csv = Path("/kaggle/working/results/tables/valendin_replication_interpretation.csv")
    if report_csv.exists():
        display(pd.read_csv(report_csv))
    else:
        print(f"Interpretation table not found at {report_csv}")

    print("\nBuilding replication diagnostics...")
    subprocess.run(
        [sys.executable, "-m", "src.evaluation.replication_diagnostics",
         "--results-dir", "/kaggle/working/results"],
        check=False,
    )
    diag_csv = Path("/kaggle/working/results/tables/valendin_replication_per_week_summary.csv")
    if diag_csv.exists():
        display(pd.read_csv(diag_csv))
    else:
        print(f"Per-week diagnostics not found at {diag_csv}")

## 4 · Stage 1 — Base LSTM (all datasets)

Runs `lstm_base_{uci,tafeng,dunnhumby}_final` from `experiments/configs_final/`.
These apply the Valendin-winning settings (Keras init, linear dense, no dropout/weight-decay, Adam eps=1e-7, patience=5) to all three non-CDNOW datasets. Set `RUN_ALL_STAGES = True` to run.

CDNOW base is handled by Stage 0.

In [ ]:
# Stage 1 — Base LSTM extension sweep (all non-CDNOW datasets, 3 seeds)
# Uses configs_final/ with Valendin-winning settings.
# Set RUN_ALL_STAGES = True to run all thesis stages.
import subprocess, sys

RUN_ALL_STAGES = True   # set False to run Stage 0 only
FINAL_CONFIG_DIR = "experiments/configs_final"

CONFIGS = [
    "lstm_base_uci_final",
    "lstm_base_tafeng_final",
    "lstm_base_dunnhumby_final",
]
if RUN_ALL_STAGES:
    subprocess.run(
        [sys.executable, "run_seeds.py",
         "--config_dir", FINAL_CONFIG_DIR,
         "--configs", *CONFIGS,
         "--seeds", "42", "7", "2024",
         "--modes", "sample",
         "--skip_existing",
         "--heartbeat_interval", "60"],
        check=False,
    )
    print("\nGroup complete: Stage 1 — Base LSTM (UCI, TaFeng, Dunnhumby)")
else:
    print("SKIP Stage 1. Set RUN_ALL_STAGES=True to run the full final sweep.")

## 5 · Stage 2 — Joint LSTM (frequency + spend, all datasets)

Adds the hurdle-lognormal spend head with Kendall multi-task loss. All four datasets on the
same Valendin-winning base settings: Adam, no dropout/weight-decay, freq_logvar_max=-1.0 guard.

In [ ]:
# Stage 2 — Joint LSTM (all 4 datasets, 3 seeds)
import subprocess, sys

CONFIGS = [
    "lstm_joint_cdnow_final",
    "lstm_joint_uci_final",
    "lstm_joint_tafeng_final",
    "lstm_joint_dunnhumby_final",
]
if RUN_ALL_STAGES:
    subprocess.run(
        [sys.executable, "run_seeds.py",
         "--config_dir", FINAL_CONFIG_DIR,
         "--configs", *CONFIGS,
         "--seeds", "42", "7", "2024",
         "--modes", "sample",
         "--skip_existing",
         "--heartbeat_interval", "60"],
        check=False,
    )
    print("\nGroup complete: Stage 2 — Joint LSTM (all datasets)")
else:
    print("SKIP Stage 2. Set RUN_ALL_STAGES=True.")

## 6 · Stage 3 — Joint Transformer (Time2Vec + sinusoidal PE, all datasets)

Transformer encoder with d_model=128 (matches LSTM hidden_size for fair comparison), n_heads=4, n_layers=2.
Time2Vec + sinusoidal PE (required by proposal). Joint frequency + spend with Kendall loss.

In [ ]:
# Stage 3 — Joint Transformer (all 4 datasets, 3 seeds)
import subprocess, sys

CONFIGS = [
    "transformer_joint_cdnow_final",
    "transformer_joint_uci_final",
    "transformer_joint_tafeng_final",
    "transformer_joint_dunnhumby_final",
]
if RUN_ALL_STAGES:
    subprocess.run(
        [sys.executable, "run_seeds.py",
         "--config_dir", FINAL_CONFIG_DIR,
         "--configs", *CONFIGS,
         "--seeds", "42", "7", "2024",
         "--modes", "sample",
         "--skip_existing",
         "--heartbeat_interval", "60"],
        check=False,
    )
    print("\nGroup complete: Stage 3 — Joint Transformer (all datasets)")
else:
    print("SKIP Stage 3. Set RUN_ALL_STAGES=True.")

## 6.5 · Replication-only diagnostics — expected and 100-scenario rescore

Re-scores strict and full-batch Base LSTM checkpoints. This is diagnostic-only: it quantifies sampling noise and transition-model underprediction without retraining and without calibration repairs.

Outputs are labeled `*_expected_rescore` and `*_sample100_rescore` and should not be used as headline Valendin replication results.

In [ ]:
# Stage 3.5 — strict/full-batch replication diagnostic rescoring (no retraining)
# No temperature fitting and no aggregate calibration: diagnostic-only, not a repair.
import subprocess, sys, glob
from pathlib import Path

RESULTS_DIR = Path("/kaggle/working/results")
CKPT_DIR    = RESULTS_DIR / "checkpoints"

RESCORE_TARGETS = [
    ("experiments/configs/lstm_base_cdnow_replication.yaml", "lstm_base_cdnow_replication_demo_pure"),
    ("experiments/configs/lstm_base_cdnow_replication_fullbatch.yaml", "lstm_base_cdnow_replication_fullbatch"),
]
SEEDS = [42, 7, 2024]
RESCORE_MODES = [
    ("expected", 30, "expected_rescore"),
    ("sample", 100, "sample100_rescore"),
]

n_done = n_skip = n_miss = 0
for config_yaml, run_prefix in RESCORE_TARGETS:
    for seed in SEEDS:
        ckpt = CKPT_DIR / f"{run_prefix}_seed{seed}_sample.pt"
        for mode, n_scenarios, suffix in RESCORE_MODES:
            out_run = f"{run_prefix}_seed{seed}_{suffix}"
            metrics_out = RESULTS_DIR / "tables" / f"{out_run}_metrics.json"

            if metrics_out.exists():
                print(f"  SKIP (exists): {out_run}")
                n_skip += 1
                continue
            if not ckpt.exists():
                print(f"  MISS (no ckpt): {ckpt.name}  — run the Valendin replication cell first")
                n_miss += 1
                continue

            print(f"  RESCORE: {out_run}")
            r = subprocess.run(
                [sys.executable, "-m", "src.evaluation.rescore",
                 "--config",     config_yaml,
                 "--checkpoint", str(ckpt),
                 "--run-name",   out_run,
                 "--mode",       mode,
                 "--n-scenarios", str(n_scenarios),
                 "--results-dir", str(RESULTS_DIR),
                 "--kaggle",
                 "--kaggle-data-root", str(Path("/kaggle/working/input")),
                ],
                check=False,
            )
            if r.returncode == 0:
                n_done += 1
            else:
                print(f"    WARNING: rescore returned {r.returncode}")

print(f"\nDiagnostic rescore complete: {n_done} done, {n_skip} skipped, {n_miss} missing checkpoints")


## 7 · Stage 4 — Extension 3 covariate ablation (Dunnhumby 80/4)

Four covariate conditions × two model types × 3 seeds = 24 runs.
- **none**: control (no covariates)
- **static**: household demographics (income_desc, household_size_desc)
- **dynamic**: campaign exposure (coupon_redemptions_per_week, campaign_exposure_flag)
- **full**: static + dynamic

SHAP attribution runs on the full models (LSTM + Transformer) in Stage 8.

In [ ]:
# Stage 4a — Extension 3 LSTM covariate ablation (none/static/dynamic/full × 3 seeds)
import subprocess, sys

CONFIGS = [
    "extension3_lstm_none_dunnhumby_final",
    "extension3_lstm_static_dunnhumby_final",
    "extension3_lstm_dynamic_dunnhumby_final",
    "extension3_lstm_full_dunnhumby_final",
]
if RUN_ALL_STAGES:
    subprocess.run(
        [sys.executable, "run_seeds.py",
         "--config_dir", FINAL_CONFIG_DIR,
         "--configs", *CONFIGS,
         "--seeds", "42", "7", "2024",
         "--modes", "sample",
         "--skip_existing",
         "--heartbeat_interval", "60"],
        check=False,
    )
    print("\nGroup complete: Stage 4a — Extension 3 LSTM (none/static/dynamic/full × 3 seeds)")
else:
    print("SKIP Stage 4a. Set RUN_ALL_STAGES=True.")

In [ ]:
# Stage 4b — Extension 3 Transformer covariate ablation (none/static/dynamic/full × 3 seeds)
import subprocess, sys

CONFIGS = [
    "extension3_transformer_none_dunnhumby_final",
    "extension3_transformer_static_dunnhumby_final",
    "extension3_transformer_dynamic_dunnhumby_final",
    "extension3_transformer_full_dunnhumby_final",
]
if RUN_ALL_STAGES:
    subprocess.run(
        [sys.executable, "run_seeds.py",
         "--config_dir", FINAL_CONFIG_DIR,
         "--configs", *CONFIGS,
         "--seeds", "42", "7", "2024",
         "--modes", "sample",
         "--skip_existing",
         "--heartbeat_interval", "60"],
        check=False,
    )
    print("\nGroup complete: Stage 4b — Extension 3 Transformer (none/static/dynamic/full × 3 seeds)")
else:
    print("SKIP Stage 4b. Set RUN_ALL_STAGES=True.")

## 8 · SHAP covariate attribution (Extension 3 full Joint LSTM)

In [ ]:
# Stage 8 — SHAP covariate attribution (Extension 3 full models)
# Runs on the best extension3_lstm_full_dunnhumby_final checkpoint (seed 42).
# Also runs on extension3_transformer_full if checkpoint is present.
import subprocess, sys, glob
from pathlib import Path

Path("/kaggle/working/results/plots").mkdir(parents=True, exist_ok=True)

for model_prefix in [
    "extension3_lstm_full_dunnhumby_final",
    "extension3_transformer_full_dunnhumby_final",
]:
    config_yaml = f"experiments/configs_final/{model_prefix}.yaml"
    ckpt_pattern = f"/kaggle/working/results/checkpoints/{model_prefix}*seed42*.pt"
    ckpts = sorted(glob.glob(ckpt_pattern))
    if not ckpts:
        print(f"No checkpoint found: {ckpt_pattern} — run the training stages first.")
        continue
    print(f"\nRunning SHAP for {model_prefix} ...")
    subprocess.run([
        sys.executable, "-m", "src.evaluation.shap_analysis",
        "--config", config_yaml,
        "--checkpoint", ckpts[-1],
        "--n_background", "100",
        "--n_explain", "200",
        "--out_dir", "/kaggle/working/results/plots",
    ], check=False)

plots = sorted(Path("/kaggle/working/results/plots").glob("*.png"))
print(f"\nSHAP plots saved: {len(plots)}")
for p in plots:
    print(f"  {p.name}")

## 9 · Benchmarks + final comparison tables & plots

Copies the committed probabilistic-benchmark results into the working results, then builds `comparison_thesis.tex`, `comparison_seeds.csv`, and the thesis figures. Every model — DL and benchmark — is scored on Valendin's **weekly** MAPE and on CLV ρ.

In [ ]:
# Copy committed benchmark results (run locally; Pareto/GGG needs R) into working results.
import shutil, glob, subprocess
from pathlib import Path

work = Path("/kaggle/working/results/tables"); work.mkdir(parents=True, exist_ok=True)
repo_tables = Path("results/tables")
copied = 0
for pat in ["pareto_nbd_*", "bgnbd_gg_*", "pareto_ggg_*"]:
    for f in glob.glob(str(repo_tables / pat)):
        shutil.copy(f, work); copied += 1
print(f"Copied {copied} committed benchmark files into {work}")

# Build comparison tables + plots (DL results + benchmarks, all on weekly MAPE).
subprocess.run(
    ["python", "-m", "src.evaluation.compare",
     "--results_dir", "/kaggle/working/results",
     "--latex", "--plots", "--seeds", "--cis",
     "--include_exploratory", "--protocol_variant", "all",
     "--include_expected"],
    check=False,
)
plots = sorted(Path("/kaggle/working/results/plots").glob("*.png")) + \
        sorted(Path("/kaggle/working/results/plots").glob("*.pdf"))
print(f"\nPlots generated: {len(plots)}")
for p in plots:
    print(f"  {p.name}")


## 10 · Archive results for download

In [ ]:
import shutil, os
from pathlib import Path

results_dir  = Path("/kaggle/working/results")
archive_stem = "/kaggle/working/results_archive"   # .zip will be appended automatically

if not results_dir.exists() or not any(results_dir.rglob("*")):
    print("No results found yet — run at least one training cell first.")
else:
    # Report what we have before archiving
    metrics_files = sorted(results_dir.rglob("*_metrics.json"))
    ckpt_files    = sorted(results_dir.rglob("*.pt"))
    print(f"Metrics files  : {len(metrics_files)}")
    print(f"Checkpoints    : {len(ckpt_files)}")
    for f in metrics_files:
        print(f"  {f.name}")

    # Build zip
    shutil.make_archive(archive_stem, "zip", results_dir)
    archive_path = Path(archive_stem + ".zip")
    size_mb = archive_path.stat().st_size / (1024 ** 2)
    print(f"\nArchive created : {archive_path}  ({size_mb:.1f} MB)")
    print("Download via    : Kaggle notebook → Output tab → results_archive.zip")